# Chuẩn hóa và khử trùng qua version

Bỏ wrapper định dạng, giữ nội dung và chuẩn hóa math/cite. BibTeX hợp nhất fields, ánh xạ citation keys theo paper/version. Full-text khử trùng chính xác theo nội dung đã làm sạch và kind; occurrence và edges vẫn giữ theo version.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from crawler.reporting import load_m2, run_table
m2 = load_m2()
manifest, stats = m2['manifest'], m2['stats']
print('Input:', m2['directory'])
print('Phạm vi:', stats.get('publications'), 'paper;', stats.get('versions'), 'version')
print('Run:', stats.get('run_id'), '; status:', stats.get('status'))


In [ ]:
dedup=pd.DataFrame(stats.get('deduplication',{})).T
display(dedup)
print('Version chưa đo được before-counts:',stats.get('versions_with_unmeasured_counts'))
print('Số citation chưa resolve:',stats.get('unresolved_cites'))
print('Số bibliography field conflicts:',stats.get('bibliography_conflicts'))
if not dedup.empty:
    dedup[['before','after']].plot.bar(title='Counts trước/sau dedup — input parser hiện tại')
    plt.tight_layout(); plt.show()
unresolved=[]
for row in manifest.to_dict('records'):
    keys=row.get('unresolved_cites') or []
    if keys:
        unresolved.append({'arxiv_id':row['arxiv_id'],'version':row['version'],'unresolved_count':len(keys),'example_keys':', '.join(keys[:8])})
display(pd.DataFrame(unresolved))
if m2['working']:
    example=m2['working'][0]
    print('Citation mappings (ví dụ):',example.get('citation_mappings',{}))
    print('Field conflicts (ví dụ):',example.get('field_conflicts',[])[:5])
    display(pd.DataFrame([{'version':row['version'],'kind':row['kind'],'raw':row['content'][:140],
                           'normalized':row['normalized_content'][:140],'element_id':row['element_id']}
                          for row in example.get('occurrences',[])[:15]]))


## Giới hạn

Số dòng trước/sau dedup lấy từ manifest. Kiểm tra citation conflicts và unresolved keys khi cùng key trỏ tới tài liệu khác nhau. Figure giữ nội dung/caption TeX; corpus không chứa ảnh nhị phân.
